# CNN Homework: Five-Class Vehicle Classification

This notebook implements the CNN stages presented in the course slides:

1. Convolution
2. ReLU activation
3. Max pooling
4. Flattening/vectorization
5. Fully connected classification
6. Five-class softmax output

The five classes are **airplanes, buses, cars, motorcycles, and ship**.

In [ ]:
from pathlib import Path
import json
import random

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

CLASS_NAMES = ["airplanes", "buses", "cars", "motorcycles", "ship"]
NUM_CLASSES = len(CLASS_NAMES)
IMAGE_SIZE = (256, 256)
BATCH_SIZE = 16
TRAIN_DIR = Path("Datasets/train/images_datasets")
TEST_DIR = Path("Datasets/test")
MODEL_PATH = Path("best_cnn_model.keras")

print("TensorFlow version:", tf.__version__)
print("Classes:", CLASS_NAMES)

## 1. Dataset validation and split

Each class has 100 source images. A 20% validation split produces:

- 80 training images per class (400 total)
- 20 validation images per class (100 total)
- 20 separate test images per class (100 total)

The validation images remain in the training folders; Keras selects them in memory.

In [ ]:
def image_count(folder):
    extensions = {".jpg", ".jpeg", ".png", ".bmp", ".gif"}
    return sum(
        item.is_file() and item.suffix.lower() in extensions
        for item in folder.iterdir()
    )


for root in (TRAIN_DIR, TEST_DIR):
    if not root.is_dir():
        raise FileNotFoundError(f"Dataset directory not found: {root}")

    for class_name in CLASS_NAMES:
        class_dir = root / class_name
        if not class_dir.is_dir():
            raise FileNotFoundError(f"Missing class directory: {class_dir}")
        count = image_count(class_dir)
        if count == 0:
            raise ValueError(f"No images found in: {class_dir}")
        print(f"{class_dir}: {count} images")

In [ ]:
train_datagen = keras.preprocessing.image.ImageDataGenerator(
    validation_split=0.20,
    rotation_range=15,
    width_shift_range=0.10,
    height_shift_range=0.10,
    zoom_range=0.15,
    horizontal_flip=True,
    fill_mode="nearest",
)

# Validation and test images are not augmented.
validation_datagen = keras.preprocessing.image.ImageDataGenerator(
    validation_split=0.20
)
test_datagen = keras.preprocessing.image.ImageDataGenerator()

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    classes=CLASS_NAMES,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="training",
    shuffle=True,
    seed=SEED,
)

validation_generator = validation_datagen.flow_from_directory(
    TRAIN_DIR,
    classes=CLASS_NAMES,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="validation",
    shuffle=False,
    seed=SEED,
)

test_generator = test_datagen.flow_from_directory(
    TEST_DIR,
    classes=CLASS_NAMES,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False,
)

assert list(train_generator.class_indices) == CLASS_NAMES
assert list(test_generator.class_indices) == CLASS_NAMES
print("Class indices:", train_generator.class_indices)

## 2. One-hot encoding

With `class_mode="categorical"`, Keras represents each label as a five-position one-hot vector. For example, the ship class is `[0, 0, 0, 0, 1]`.

In [ ]:
sample_images, sample_labels = next(train_generator)

print("One-hot label shape:", sample_labels.shape)
print("Example one-hot label:", sample_labels[0])
print("Decoded class:", CLASS_NAMES[int(np.argmax(sample_labels[0]))])

plt.figure(figsize=(12, 8))
for index in range(min(12, len(sample_images))):
    plt.subplot(3, 4, index + 1)
    plt.imshow(sample_images[index].astype("uint8"))
    plt.title(CLASS_NAMES[int(np.argmax(sample_labels[index]))])
    plt.axis("off")
plt.tight_layout()
plt.show()

train_generator.reset()

## 3. CNN architecture

Each convolutional block performs **convolution + ReLU + max pooling**. Early layers learn low-level features such as edges and colors. Deeper layers learn higher-level vehicle shapes.

Five pooling operations reduce the feature-map dimensions before `Flatten`. This keeps the fully connected layer reasonably small while explicitly demonstrating the vectorization stage in the slides.

In [ ]:
model = keras.Sequential(
    [
        keras.Input(shape=IMAGE_SIZE + (3,), name="input_image"),
        layers.Rescaling(1.0 / 255.0, name="normalize_pixels"),

        layers.Conv2D(32, (3, 3), padding="same", activation="relu", name="conv_1"),
        layers.MaxPooling2D((2, 2), name="pool_1"),

        layers.Conv2D(64, (3, 3), padding="same", activation="relu", name="conv_2"),
        layers.MaxPooling2D((2, 2), name="pool_2"),

        layers.Conv2D(128, (3, 3), padding="same", activation="relu", name="conv_3"),
        layers.MaxPooling2D((2, 2), name="pool_3"),

        layers.Conv2D(192, (3, 3), padding="same", activation="relu", name="conv_4"),
        layers.MaxPooling2D((2, 2), name="pool_4"),

        layers.Conv2D(256, (3, 3), padding="same", activation="relu", name="conv_5"),
        layers.MaxPooling2D((2, 2), name="pool_5"),

        layers.Flatten(name="flatten"),
        layers.Dense(128, activation="relu", name="fully_connected"),
        layers.Dropout(0.40, name="dropout"),
        layers.Dense(NUM_CLASSES, activation="softmax", name="softmax_output"),
    ],
    name="five_class_cnn",
)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.0001),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

model.summary()

In [ ]:
print(f"{'Layer':<24} {'Output shape'}")
print("-" * 50)
for layer in model.layers:
    print(f"{layer.name:<24} {str(layer.output.shape)}")

## 4. CNN training

During the forward pass, the CNN calculates five softmax probabilities. Categorical cross-entropy compares those probabilities with the one-hot label. Backpropagation calculates gradients, and Adam updates the convolution kernels, biases, and fully connected weights.

Early stopping prevents unnecessary overfitting, while model checkpointing preserves the best validation model.

In [ ]:
callbacks = [
    keras.callbacks.ModelCheckpoint(
        MODEL_PATH,
        monitor="val_loss",
        save_best_only=True,
        verbose=1,
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=2,
        min_lr=1e-6,
        verbose=1,
    ),
]

history = model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=30,
    callbacks=callbacks,
    verbose=2,
)

model = keras.models.load_model(MODEL_PATH)
with open("class_names.json", "w", encoding="utf-8") as file:
    json.dump(CLASS_NAMES, file, indent=2)

print(f"Best model saved to: {MODEL_PATH}")

## 5. Training and validation simulation results

In [ ]:
epochs_run = range(1, len(history.history["loss"]) + 1)

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(epochs_run, history.history["accuracy"], label="Training accuracy")
plt.plot(epochs_run, history.history["val_accuracy"], label="Validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training and validation accuracy")
plt.grid(alpha=0.3)
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs_run, history.history["loss"], label="Training loss")
plt.plot(epochs_run, history.history["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Categorical cross-entropy")
plt.title("Training and validation loss")
plt.grid(alpha=0.3)
plt.legend()

plt.tight_layout()
plt.show()

## 6. Final evaluation on unseen test images

In [ ]:
test_generator.reset()
test_loss, test_accuracy = model.evaluate(test_generator, verbose=0)
probabilities = model.predict(test_generator, verbose=0)
predicted_classes = np.argmax(probabilities, axis=1)
true_classes = test_generator.classes

print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.2%}\n")
print(
    classification_report(
        true_classes,
        predicted_classes,
        labels=range(NUM_CLASSES),
        target_names=CLASS_NAMES,
        digits=4,
        zero_division=0,
    )
)

ConfusionMatrixDisplay.from_predictions(
    true_classes,
    predicted_classes,
    labels=range(NUM_CLASSES),
    display_labels=CLASS_NAMES,
    cmap="Blues",
    xticks_rotation=45,
)
plt.title("Five-class test confusion matrix")
plt.tight_layout()
plt.show()

## 7. Sample test predictions

In [ ]:
test_generator.reset()
test_images, test_labels = next(test_generator)
batch_probabilities = model.predict(test_images, verbose=0)

plt.figure(figsize=(12, 10))
for index in range(min(15, len(test_images))):
    true_index = int(np.argmax(test_labels[index]))
    predicted_index = int(np.argmax(batch_probabilities[index]))
    confidence = float(batch_probabilities[index, predicted_index])
    color = "green" if true_index == predicted_index else "red"

    plt.subplot(3, 5, index + 1)
    plt.imshow(test_images[index].astype("uint8"))
    plt.title(
        f"True: {CLASS_NAMES[true_index]}\n"
        f"Pred: {CLASS_NAMES[predicted_index]}\n"
        f"{confidence:.1%}",
        color=color,
        fontsize=8,
    )
    plt.axis("off")

plt.tight_layout()
plt.show()
test_generator.reset()

## 8. Predict one image and display softmax probabilities

In [ ]:
def test_single_image(image_path, model_path=MODEL_PATH):
    image_path = Path(image_path)
    model_path = Path(model_path)

    if not image_path.is_file():
        raise FileNotFoundError(f"Image not found: {image_path}")
    if not model_path.is_file():
        raise FileNotFoundError(f"Model not found: {model_path}. Train the model first.")

    prediction_model = keras.models.load_model(model_path)
    image = keras.utils.load_img(image_path, target_size=IMAGE_SIZE)
    image_array = keras.utils.img_to_array(image)
    image_batch = np.expand_dims(image_array, axis=0)

    probabilities = prediction_model.predict(image_batch, verbose=0)[0]
    predicted_index = int(np.argmax(probabilities))
    predicted_class = CLASS_NAMES[predicted_index]
    confidence = float(probabilities[predicted_index])

    plt.figure(figsize=(14, 5))
    plt.subplot(1, 2, 1)
    plt.imshow(image)
    plt.title(f"Predicted: {predicted_class}\nConfidence: {confidence:.2%}")
    plt.axis("off")

    plt.subplot(1, 2, 2)
    colors = ["crimson" if i == predicted_index else "steelblue" for i in range(NUM_CLASSES)]
    bars = plt.bar(CLASS_NAMES, probabilities * 100, color=colors)
    plt.ylabel("Probability (%)")
    plt.title("Softmax class probabilities")
    plt.ylim(0, 105)
    plt.xticks(rotation=35, ha="right")
    for bar, probability in zip(bars, probabilities):
        plt.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 1,
            f"{probability:.1%}",
            ha="center",
        )
    plt.tight_layout()
    plt.show()

    print(f"Image: {image_path.name}")
    print(f"Predicted class: {predicted_class}")
    print(f"Confidence: {confidence:.2%}")
    for label, probability in zip(CLASS_NAMES, probabilities):
        print(f"{label:<12}: {probability:.2%}")

    return predicted_class, confidence, probabilities

In [ ]:
test_single_image("Datasets/test/ship/ship_005.jpg")

## 9. Conclusion

After running the notebook, summarize the actual results here:

- Compare training and validation accuracy.
- State the final test accuracy and loss.
- Identify the strongest and weakest classes from the classification report.
- Use the confusion matrix to explain which vehicle classes are confused.
- Discuss whether more images, improved image quality, or different augmentation could improve the CNN.

Do not write a final conclusion until the simulation results have been generated.